<a href="https://www.kaggle.com/code/jmrb103/predicting-telecon-churn?scriptVersionId=302504918" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Telco Costumer Churn Preditions

I picked this model to apply my Data Science skills in a classic buisness prblem. 

## Overview

Customer churn occurs when a costumer has left the company. For a telecom company it is crucial to predict who is likely to leave (churn) given that it is more expensive to acquire new costumers than to keep existing ones.

## The Goal

We will explore and analyze Teleco costumer data to predict what customers are likely to leave.

In [ ]:
# Load libraries
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

## Loading and Inspecting the Data

We will inspect the data and fix any error in the raw data.

In [ ]:
# Load dataset
df = pd.read_csv(
    '/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv',
    # nrows=2000
)
print(df.info())
df.head()

In [ ]:
print(df.describe())
print(df.isna().sum())

In [ ]:
# Transform 'TotalCharges' to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Finding NaNs
print(f"Total NaNs: {df['TotalCharges'].isnull().sum()}")

# Removing NaN
df = df.dropna(subset='TotalCharges')

df.isna().sum()

In [ ]:
# Find uniques
df.nunique()

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

## EDA and Visualization

What I want to do here is to explore the variables in the table to find the most relevant features that may influence `Churn`

In [ ]:
# Set plotting style
sns.set_style('darkgrid')
sns.set_palette('muted')

# Visualize churn distribution
sns.countplot(data=df, x='Churn', hue='Churn')
plt.title('Churn distribution (0=stayes, 1=left)')
plt.show()

We can appreciate that customers stay (0) almost 3x than the numer of customers that left (1)

In [ ]:
sns.histplot(x='tenure', hue='Churn', data=df)
plt.title('Tenure Distribution by Churn')
plt.xlabel('Months with Teleco')
plt.show()

In [ ]:
df.select_dtypes(include="number").columns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

i = 0
df_num_cols = df.select_dtypes(include="number").columns
for ax in axes:
    # First column
    sns.histplot(data=df, x=df_num_cols[i], hue='Churn', ax=ax[0])
    i +=1
    # Second column
    sns.histplot(data=df, x=df_num_cols[i], hue='Churn', ax=ax[1])
    i += 1

fig.suptitle('Distributions by Churn  (0=stayes, 1=left)')
plt.tight_layout()

Let us explore the distribution of the other features by churn.

**Senior Citizen**

We can see in this plot that the majority of customers in Telecon are non-senior, and that it is the younger customers that stay with Telecon.

**Tenure**

The customers that have been with the company the longest seem to the the least likely to leave. In other words: long-time customers are the most loyal to Teleco.

**Monthly Charges**

The costumers that stay with Telecon are the ones that receive < 20 charges per month. The likelelyhood that customers leave Telecon grows if they receive > 60 charges per month.

**Total Charges**

This quantity at first sight doesn't throw any trend that may inform us if this causes customers to leave Telecon.

In [ ]:
df_obj_cols = df.select_dtypes(include="object").columns[1:]
obj_cols_len = len(df_obj_cols)

fig, axes = plt.subplots(obj_cols_len, figsize=(8, 40))

for ax, col in zip(axes, df_obj_cols):
    sns.countplot(data=df, x=col, hue='Churn', ax=ax)

plt.tight_layout()

_Note_: I will describe the most relevant plots from the above analysis.

**Online Backup**

There seems to be a rise in customers leaving Telecon if they do not have online backup.

**Device Protection**

The population of churns that have no device protection. If customers have device protection, tend to not leave Telecon.

**Tech Support**

Customers without tech support tend to leave the company as opposed to those with technical support, or no internet service

**Contract**

Te distribution of Mont-to-Month show that show that in this group the population of customers that leave Telecon is higher.

**Payment Method**

In this class, the group of customers that pay through electronic check show that the population of that leave Telecon is higher.

## Preprocessing

Here we will prepare the data for our Machine Learning model(s). This means converting `object` columns into numbers, and scaling our numerical features.

In [ ]:
# Removing customerID
df_clean = df.drop(columns=['customerID'])

# One-hot Encoding
df_encoded = pd.get_dummies(df_clean, columns=df_obj_cols, drop_first=True)

# Define variables and split
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model Building

We will train several models 

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(),
    'K-nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=123),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=123),
    'SVM': SVC(kernel='rbf', probability=True, random_state=123),
    'Gradient Boost': GradientBoostingClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        loss='log_loss',
        random_state=123
    ),
    'XGBoost': XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=123
    )
}

# Re-defining dictionary with Pipelines
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ('scaler', StandardScaler()),  # Step 1: Scale the data
        ('classifier', model)          # Step 2: Run the model
    ])

model_scores = {}

print('Initiating Model Training')

for name, pipe in pipelines.items():
    print(f"Training {name}...")
    
    # Fit the pipeline (Scales data + Trains model)
    pipe.fit(X_train, y_train)
    
    # Get predictions
    y_pred = pipe.predict(X_test)
    y_probs = pipe.predict_proba(X_test)[:, 1] # Needed for ROC AUC
    
    # Calculate all metrics
    model_scores[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'roc_curve': roc_curve(y_test, y_probs),
        'roc_auc': roc_auc_score(y_test, y_probs)
    }
    
print("\n-> Training Complete!")

## Evaluation

Now that we have the performance of each model, let us compare and contrast to pick the best model for our data.

In [ ]:
# Convert model_scores dict to the DataFrame
results_df = pd.DataFrame.from_dict(model_scores, orient='index').reset_index()
results_df.rename(columns={'index': 'model'}, inplace=True)

# Display the numeric summary (sorted by F1)
summary_cols = ['model', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
print(results_df[summary_cols].sort_values(by='f1_score', ascending=False))
print("\n-> Evaluation Complete!")

In [ ]:
# Transforming dataframe from wide to long
melt_df = results_df.melt(id_vars='model', var_name='metric', value_name='value')
melt_df = melt_df[~melt_df['metric'].isin(['confusion_matrix', 'roc_curve'])]

# Plotting scores per model
plt.figure(figsize=(10, 15))
sns.barplot(x='value', y='model', hue='metric', data=melt_df)

plt.title('Model Performance Comparison (All Metrics)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

From the above comparison of metrics accross several models, the model that has the best scores is **Logistic Regression**.

Let us see further results results from the winning model.

In [ ]:
best_model = 'Logistic Regression'

# Plotting Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(model_scores[best_model]['confusion_matrix'], annot=True, fmt='d', cmap='viridis')
plt.title(f'Confusion Matrix ({best_model})')
plt.ylabel('Actual Truth')
plt.xlabel('Prediction')
plt.show()

In [ ]:
# Plotting ROC curves
plt.figure(figsize=(6, 5))
for model, score in model_scores.items():
    fpr, tpr, _ = score['roc_curve']
    plt.plot(fpr, tpr, label=model)
    if model == best_model:
        plt.fill_between(fpr, tpr, alpha=0.5)
plt.title('ROC Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.show()

### Feature importance

Let us compare the importance of each feature to our winning model. The importance of each feature will guide us to understand what drives the clients to leave Telecon.

Just for educational purposes we will compare the feature importance with XGBoost scores.

In [ ]:
# Logistic Regression coefficients
lr_model = pipelines['Logistic Regression'].named_steps['classifier']
lr_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lr_model.coef_[0]
})
lr_df['abs_importance'] = lr_df['importance'].abs()
lr_df = lr_df.sort_values(by='abs_importance', ascending=False)

# XGBoost feature importances
rf_model = pipelines['XGBoost'].named_steps['classifier']
rf_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values(by='importance', ascending=False)

# 3. Create the Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 18))

# Plot Logistic Regression (Directional)
sns.barplot(
    data=lr_df,
    x='importance',
    y='feature',
    hue='feature',
    legend=False,
    ax=ax1,
    palette='RdBu_r'
)
ax1.set_title('Logistic Regression\n(Directional Coefficients)')
ax1.axvline(0, color='black', lw=1)

# Plot XGBoost (Magnitude)
sns.barplot(
    data=rf_df,
    x='importance',
    y='feature',
    hue='feature',
    legend=False,
    ax=ax2,
    palette='viridis'
)
ax2.set_title('XGBoost\n(Gini Importance)')

plt.tight_layout()
plt.show()

According to our Logistic Regression model, the **top 3** features that impact more in the clients to leave the company are:

- Tenure
- Monthly charges
- Fiber optic internet service

 ## Conclusion

 From our evaluation above we can conclude that the best model to make predictions is: **Logit Regression**, followed by Gradient Boost and XGBoost, which can be further tuned.

Telecon can use this model to scan all current customers and predict what clients are more likely to leave.

According to this model, those clients flagged as "high risk of churn" can get offers of a discount if they move to a longer term contract, or offer them a free tech support service to check the fiber optic internet service.

## TODO

- [ ] Apply one-hot encoding, and label encoding accordingly, instead of applying on-hot to every categorical features.
- [ ] Pick the top 3 models and use cross-validation method to find the best parameters with the bset model.